In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\20254833\Repositories\llm_zoomcamp\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\20254833\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
from tqdm.auto import tqdm
import numpy as np

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
v1.dot(dv)

np.float32(0.32332402)

In [3]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2.dot(dv)

np.float32(0.019730477)

In [4]:
from ingest import load_faq_data

documents = load_faq_data()

Generating embeddings

In [8]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

print(len(vectors))

X = np.array(vectors)
X.shape

  0%|          | 0/28 [00:00<?, ?it/s]

1368


Scoring documents

In [12]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)
# This is matrix-vector multiplication
scores = X.dot(v_query)
# Same thing with for loop
scores = [v_query.dot(X[i]) for i in range(len(X))]  # slower


In [18]:
scores = X.dot(v_query)
type(scores)

numpy.ndarray

Best match

In [13]:
idx = np.argmax(scores)
idx, scores[idx]
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

Top 5 results

In [ ]:
top5 = np.argsort(scores)[-5:]
# reverse them to get the highest first
top5 = top5[::-1]
top5
# Same but in one line
top5 = np.argsort(-scores)[:5]  # Works only if socres type is numpy.ndarray
scores[top5]

In [21]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294094
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.75793695
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relat

## Vector Search with minsearch

In [22]:
from minsearch import VectorSearch

# Indexing the documents for vector search
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [23]:
# Under the hood, computes the dott product between the query vector and all the document vectors.
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [24]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

Filtering by course

In [26]:
# Filter by keywords fields
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

## Using RAGBase

In module 1, we built a RAG pipeline with three steps:

```python

def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

```

In [ ]:
# Create OpenAI client
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

# Download and index the data
documents = load_faq_data()
index = build_index(documents)

# Use the RAGBase class
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

# Ask still using keyword search
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while the program is still accepting submissions.'

In [28]:
# We can't pass vindex to RAG. Vector search need the query as a vector
# We need to override search to encode the query before searching
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
# Using vector search (close to what we got with keyword search)
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join and start learning, even if the program has already begun. If you want a certificate, make sure to submit your project while submissions are still being accepted.'

It works but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force

So far we have done is exact nearest neighbor (NN) search. We score the query against every document and pick the top ones. 

Approximate nearest neighbor (ANN) search is faster, instead of comparing against everything. it first narrows down to a region of likely matches. Then scores only within that region

## Vector Search with sqlitesearch

- Persistent text search
- Stores vector in SQLite and uses ANN strategies for retrieval.

In [30]:
from sqlitesearch import VectorSearchIndex

# Creating the index
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"  # Where the index is saved
)

# Indexing the documents for vector search
vs_index.fit(vectors, documents)

# Search: encode the query into a vector and then search
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': 'c207b8614e',
  'course': 'data-engineering-zoomcamp',
 

In [33]:
# Filtering by course
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# Close the connection when done
vs_index.close()